# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You'll learn how to inspect the Croissant schema, enumerate record sets and fields, extract data, and conduct basic exploratory analysis.

### Dataset Source
Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields for schema exploration.

All entities (record sets, fields/columns) are referenced by their `@id`.

Let's list each record set and its fields/columns.

In [ ]:
from pprint import pprint

# List all record sets and their fields/columns
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rset in record_sets:
        print(f"RecordSet name: {rset.name}")
        print(f"  @id: {rset.id}")
        print(f"  Description: {rset.description}")
        print("  Fields/Columns:")
        for field in rset.fields:
            print(f"    {field.name} (@id: {field.id}) [type: {field.data_type}]")
        print("-")
    # Save a list of record set @ids for later usage
    record_set_ids = [rset.id for rset in record_sets]

# For datasets with only a single record set, you can access it e.g. `record_sets[0]`
# Print first 2 records (if available) for a record set for preview
if record_sets:
    record_set0_id = record_sets[0].id
    print(f"\nExample records from {record_set0_id}:")
    for idx, record in enumerate(dataset.records(record_set=record_set0_id)):
        pprint(record)
        if idx >= 1: break

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis.

All record set and field references use their `@id`s.

The following cell will load all available record sets as DataFrames.

In [ ]:
# Extract data for each record set
dataframes = {}
if record_sets:
    for rset in record_sets:
        print(f"Loading record set: {rset.name} (@id: {rset.id})")
        records = list(dataset.records(record_set=rset.id))
        df = pd.DataFrame(records)
        dataframes[rset.id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Number of records: {len(df)}\n")

    # Choose the first record set for further analysis
    first_record_set_id = record_sets[0].id
    df = dataframes[first_record_set_id]
    print(f"Schema for {first_record_set_id}:\n", df.dtypes)
    print(df.head())
else:
    print("No record sets present to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data using field `@id`s from the previous overview.

**All field and record set references use their `@id`.**

In [ ]:
import numpy as np

# Use first record set if present
if record_sets:
    record_set_id = first_record_set_id
    df = dataframes[record_set_id]

    # Suggest a numeric field to analyze, e.g. molecular_weight or coverage_pc, by scanning field definitions
    # Display all numeric fields first
    numeric_fields = []
    for field in record_sets[0].fields:
        dtype = str(field.data_type).lower() if field.data_type else None
        if dtype and any(x in dtype for x in ["float", "number", "int"]):
            print(f"Numeric field: {field.name} (@id: {field.id}) type: {field.data_type}")
            numeric_fields.append(field.id)

    # If a numeric field exists, perform filtering, normalization, grouping
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"\nUsing numeric field '{numeric_field_id}' for analysis.")

        # Choose a reasonable threshold (e.g. mean)
        threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns else 0
        print(f"Threshold: {threshold}")
        
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
        print(filtered_df.head())

        # Normalize
        normcol = f"{numeric_field_id}_normalized"
        filtered_df[normcol] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized column '{numeric_field_id}' added.")
        print(filtered_df[[numeric_field_id, normcol]].head())

        # Find a categorical/grouping field, e.g., accession or description
        group_fields = [f.id for f in record_sets[0].fields if f.data_type == 'Text' or f.data_type == 'schema:Text']
        group_field = group_fields[0] if group_fields else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No group field detected for grouping.")
    else:
        print("No numeric field found for EDA analysis.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field after filtering and normalization.

All visualizations use variables/fields accessed by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_fields:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Original distribution
    sns.histplot(df[numeric_field_id].dropna(), bins=30, ax=axes[0], kde=True)
    axes[0].set_title(f"Original distribution: {numeric_field_id}")
    axes[0].set_xlabel(numeric_field_id)

    # Normalized
    sns.histplot(filtered_df[normcol].dropna(), bins=30, ax=axes[1], kde=True, color='orange')
    axes[1].set_title(f"Normalized: {numeric_field_id}")
    axes[1].set_xlabel(normcol)

    plt.tight_layout()
    plt.show()
else:
    print("No suitable numeric fields to visualize.")

## 6. Conclusion

- We demonstrated loading a Croissant dataset and exploring its metadata, record sets, and field definitions using their `@id`s.
- Data was programmatically extracted from all available record sets and converted into DataFrames.
- Numeric fields were filtered, normalized, grouped, and visualized to reveal their distribution and support further analysis.
- This workflow can be adapted for any Croissant-compliant dataset by inspecting the schema and referencing `@id`s for all data elements.